In [0]:
from delta.tables import DeltaTable
import os
import sys

# Ensure Python can find utils.py from the parent directory
project_dir = "/Workspace/Users/ranayuvrajsingh1111@gmail.com/Olist ecomerce ETL"
if project_dir not in sys.path:
  sys.path.append(project_dir)

import utils
from pyspark.sql import functions as F

# ==========================================
# 1. PARAMETERIZATION (Databricks Widget)
# ==========================================
dbutils.widgets.text("batch_name", "batch_1", "Target Batch Name")
batch_name = dbutils.widgets.get("batch_name")

print(f"Processing Silver Order Payments for Batch: {batch_name}")

# ==========================================
# 2. EXTRACT INCOMING DATA
# ==========================================
df_payments_raw = spark.read.table(
    "ecommerce_lakehouse.bronze.bronze_olist_order_payments"
).filter(F.col("_batch_id") == batch_name)

# ==========================================
# 3. TRANSFORMATIONS & DATA HYGIENE
# ==========================================
df_cleaned = df_payments_raw.select(
    F.trim(F.col("order_id")).alias("order_id"),
    F.col("payment_sequential").cast("int").alias("payment_sequential"),
    F.lower(F.trim(F.col("payment_type"))).alias("payment_type"),
    F.coalesce(F.col("payment_installments").cast("int"), F.lit(1)).alias(
        "payment_installments"
    ),
    F.coalesce(F.col("payment_value").cast("double"), F.lit(0.0)).alias(
        "payment_value"
    ),
).dropDuplicates(["order_id", "payment_sequential"])

# Compute Row Hash using SHA-256 for change detection / upserts
df_hashed = df_cleaned.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("payment_type"), F.lit("")),
            F.coalesce(F.col("payment_installments").cast("string"), F.lit("")),
            F.coalesce(F.col("payment_value").cast("string"), F.lit("")),
        ),
        256,
    ),
)

# ==========================================
# 4. TARGET TABLE INITIALIZATION OR UPSERT
# ==========================================
target_table_name = "ecommerce_lakehouse.silver.silver_order_payments"
table_exists = spark.catalog.tableExists(target_table_name)
current_timestamp = F.current_timestamp()

if not table_exists:
  print("Target table does not exist. Initializing silver_order_payments...")

  df_transformed = df_hashed.repartition(1).select(
      "order_id",
      "payment_sequential",
      "payment_type",
      "payment_installments",
      "payment_value",
      "row_hash",
      current_timestamp.alias("_etl_inserted_at"),
      current_timestamp.alias("_etl_updated_at"),
      F.lit(False).alias("_delete_flag"),
  )

  df_transformed.write.format("delta").mode("overwrite").saveAsTable(
      target_table_name
  )
  print("Successfully initialized silver_order_payments table!")

else:
  print("Table exists. Executing incremental upsert merge...")

  deltaTable = DeltaTable.forName(spark, target_table_name)

  deltaTable.alias("target").merge(
      source=df_hashed.alias("source"),
      condition=(
          "target.order_id = source.order_id AND target.payment_sequential ="
          " source.payment_sequential"
      ),
  ).whenMatchedUpdate(
      condition="target.row_hash <> source.row_hash",
      set={
          "payment_type": F.col("source.payment_type"),
          "payment_installments": F.col("source.payment_installments"),
          "payment_value": F.col("source.payment_value"),
          "row_hash": F.col("source.row_hash"),
          "_etl_updated_at": current_timestamp,
      },
  ).whenNotMatchedInsert(
      values={
          "order_id": F.col("source.order_id"),
          "payment_sequential": F.col("source.payment_sequential"),
          "payment_type": F.col("source.payment_type"),
          "payment_installments": F.col("source.payment_installments"),
          "payment_value": F.col("source.payment_value"),
          "row_hash": F.col("source.row_hash"),
          "_etl_inserted_at": current_timestamp,
          "_etl_updated_at": current_timestamp,
          "_delete_flag": F.lit(False),
      },
  ).execute()

  print("Silver Order Payments processing completed successfully!")

Processing Silver Order Payments for Batch: batch_1
Target table does not exist. Initializing silver_order_payments...
Successfully initialized silver_order_payments table!
